# Stage II (vessels) - 06. Training SwinHRUNetPP

This notebook trains the SwinHRUNetPP vessel segmenter of HierarchiRetina (Section *Stage II:
SwinHRUNetPP (vessels)* of the paper). The procedure has six steps:

1. **Configuration** - paths and seed.
2. **Dataset audit** - pair images with vessel masks and summarise resolution, vessel fraction and
   black-border fraction.
3. **Split** - the 288-image pool (DRIVE 40, STARE 20, HRF 30, MAPLES-DR 198) is split at random
   into 230 / 28 / 30 training / validation / test images (seed 42).
4. **FOV-aware pipeline** - field-of-view (FOV) mask, CLAHE + green enhancement, square FOV crop,
   512 x 512 resizing, augmentation; the vessel prior inside the FOV.
5. **Model and loss** - SwinHRUNetPP (110.2 M parameters) and the FOV-restricted loss
   0.25 BCE (w = 5.9) + 0.35 Dice + 0.30 focal Tversky + 0.10 clDice with deep supervision.
6. **Training** - AdamW, encoder frozen for 3 epochs, warm-up + cosine schedule, EMA 0.999; the
   checkpoint with the best validation Dice is kept together with its Youden-J threshold.

Requires `pip install -e .` from the repository root and a CUDA GPU for practical run times.

## Step 1. Configuration

In [ ]:
from pathlib import Path

# Expected layout. Images and masks are paired by file stem; a mask may carry one of the
# suffixes stripped by `vessel_data.norm_stem` ("_mask", "_seg", "_label", "_gt", "_vessel", ...).
#   data/vessel_dataset/image/<name>.<png|jpg|tif|ppm|...>   288 fundus images
#   data/vessel_dataset/mask/<name>[_mask].<ext>             binary vessel masks (vessel > 127)
DATA_ROOT = Path("data/vessel_dataset")
IMG_DIR = DATA_ROOT / "image"
MASK_DIR = DATA_ROOT / "mask"

CKPT_DIR = Path("checkpoints/stage2/swinhrunetpp")          # best_dice.pth, latest.pth
SPLIT_CSV = Path("outputs/stage2/vessel/vessel_split.csv")  # written here, read by notebook 07

SEED = 42
RESUME = True   # continue from CKPT_DIR/latest.pth if it exists

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from hierarchiretina.stage2 import vessel_data as vd
from hierarchiretina.stage2.engine_vessel import TRAIN_CONFIG, fit
from hierarchiretina.stage2.swinhrunetpp import (IMG_SIZE, build_swinhrunetpp, count_parameters,
                                                 init_head_bias)
from hierarchiretina.stage2.vessel_losses import CombinedVesselLoss

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Step 2. Dataset audit

Images and masks are paired by normalised file stem. The audit reports the resolution range, the
vessel fraction of the whole image and the fraction of near-black border pixels. A mean border
fraction above 10 % switches on the square crop to the FOV bounding box (it is on for the pooled
vessel data).

In [ ]:
matched, miss_mask, miss_img = vd.load_dataset_files(IMG_DIR, MASK_DIR)
print(f"Matched pairs: {len(matched)} (paper: 288) | images without mask: {len(miss_mask)} "
      f"| masks without image: {len(miss_img)}")
audit_df, audit = vd.audit_dataset(matched)
CROP_RETINA = audit["crop_retina"]
pd.Series(audit, name="value").to_frame()

## Step 3. Split (230 / 28 / 30)

`split_pairs` replays the random-number sequence of the original notebook (seed 42), which is
reproduced only if the files are listed in the same order (upper-cased file-name order, as on the
original NTFS volume). The split is saved to `SPLIT_CSV` so that the evaluation notebook uses
exactly the same test images.

In [ ]:
train_pairs, val_pairs, test_pairs = vd.split_pairs(matched, seed=SEED)
vd.save_split_csv(SPLIT_CSV, train_pairs, val_pairs, test_pairs)
print(f"train {len(train_pairs)} | val {len(val_pairs)} | test {len(test_pairs)}  -> {SPLIT_CSV}")

## Step 4. FOV-aware data pipeline

Every sample yields the image, the vessel mask (restricted to the FOV) and the FOV mask itself;
the FOV mask is passed to every loss term and metric. The positive weight of the BCE term equals
the background-to-vessel ratio inside the FOV. The notebook estimates the vessel fraction on the
first 50 augmented training samples; the reported run used 14.45 % (positive weight 5.9), which
is used below so that the final configuration is reproduced exactly.

In [ ]:
train_ds = vd.RetinalVesselDataset(train_pairs, IMG_SIZE, vd.get_train_transform(), CROP_RETINA)
val_ds = vd.RetinalVesselDataset(val_pairs, IMG_SIZE, vd.get_val_transform(), CROP_RETINA)

kw = dict(pin_memory=DEVICE.type == "cuda", num_workers=0)
train_loader = DataLoader(train_ds, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True,
                          drop_last=True, **kw)
val_loader = DataLoader(val_ds, batch_size=TRAIN_CONFIG["val_batch_size"], shuffle=False, **kw)

estimated_ratio = vd.compute_fov_vessel_ratio(train_ds, n=50)
VESSEL_RATIO = vd.FOV_VESSEL_RATIO          # 0.1445, value of the reported run
POS_WEIGHT = vd.pos_weight_from_ratio(VESSEL_RATIO)
print(f"Vessel fraction inside FOV: estimated {100 * estimated_ratio:.2f} % | "
      f"used {100 * VESSEL_RATIO:.2f} % -> BCE pos_weight {POS_WEIGHT:.2f}")

## Step 5. Model and FOV-restricted loss

SwinHRUNetPP: SwinV2-B encoder (window 8, ImageNet-1k weights from timm) in parallel with a
high-resolution CNN branch, two fusion modules, four attention-gated decoder blocks, a
multi-dilation vessel-refinement block and an elongated-kernel topology block. The four output
biases are initialised to the log-odds of the vessel prior.

In [ ]:
model = build_swinhrunetpp(pretrained=True, img_size=IMG_SIZE).to(DEVICE)
bias = init_head_bias(model, VESSEL_RATIO)
loss_fn = CombinedVesselLoss(pos_weight=POS_WEIGHT)
print(f"Parameters: {count_parameters(model) / 1e6:.1f} M (paper: 110.2 M) | head bias {bias:.3f}")

## Step 6. Training

Up to 200 epochs; batch 4 with gradient accumulation 4 (effective batch 16); AdamW (decoder
2e-4, encoder 0.1x, weight decay 1e-4), encoder frozen for the first 3 epochs, 5 warm-up epochs
then cosine decay to 1e-6; gradient clipping 1.0; mixed precision; EMA 0.999 updated once per
epoch. Validation uses the live model and the Youden-J threshold inside the FOV (clipped to
[0.05, 0.8]); `best_dice.pth` stores the best-Dice epoch (epoch 53 in the reported run,
threshold 0.46). `TRAIN_CONFIG["patience"]` is the early-stopping patience of the original
notebook; training resumes from `latest.pth` when the cell is run again.

In [ ]:
history, best = fit(model, train_loader, val_loader, loss_fn, DEVICE, CKPT_DIR, TRAIN_CONFIG,
                    resume=RESUME)
hist_df = pd.DataFrame(history)
hist_df.index = hist_df.index + 1
hist_df.index.name = "epoch"
hist_df.loc[[best["epoch"] + 1], ["dice", "precision", "recall", "auc", "opt_thr"]]